# Quantum Many-Body Scarring — Néel State Revivals

> 🚀 **Try Maestro GPU mode with a free trial.**
> Sign up at **[maestro.qoroquantum.net](https://maestro.qoroquantum.net)** — no credit card required.

## Why GPU Mode?

Scarring simulations need **60+ Trotter steps** at moderate-to-high bond dimension. A 64-atom chain at χ=64 means **hours on CPU**. GPU mode cuts this to minutes.

## Step 0 — Setup

```bash
pip install qoro-maestro matplotlib numpy
```

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import maestro

from scarring_demo import (
    build_pxp_circuit,
    build_z_observables,
    compute_staggered_magnetization,
)

---

## Phase 1 — Local (CPU, 32 atoms)

32 atoms, χ=32, 30 Trotter steps. See the first revival oscillations emerge on CPU.

In [ ]:
N_ATOMS_CPU = 32
CHI_CPU = 32
N_STEPS_CPU = 30
DT = 0.2
OMEGA = 1.0
V = 20.0  # V/Ω = 20 → strong blockade ≈ PXP

print(f"Phase 1: {N_ATOMS_CPU} atoms, χ={CHI_CPU}, {N_STEPS_CPU} steps (CPU)")
print(f"V/Ω = {V/OMEGA:.0f} (PXP limit)")

In [ ]:
observables_cpu = build_z_observables(N_ATOMS_CPU)
times_cpu = [0.0]
mag_cpu = [1.0]  # M(0) = +1 for Néel state

t0 = time.time()
for step in range(1, N_STEPS_CPU + 1):
    qc = build_pxp_circuit(N_ATOMS_CPU, OMEGA, V, DT, step)
    res = qc.estimate(
        simulator_type=maestro.SimulatorType.QCSim,
        simulation_type=maestro.SimulationType.MatrixProductState,
        observables=observables_cpu,
        max_bond_dimension=CHI_CPU,
    )
    m = compute_staggered_magnetization(res['expectation_values'], N_ATOMS_CPU)
    times_cpu.append(step * DT)
    mag_cpu.append(m)
    if step % 10 == 0:
        print(f"  step {step:3d}  t={step*DT:.1f}  M(t)={m:.4f}")

cpu_time = time.time() - t0
print(f"\n✅ Phase 1 complete in {cpu_time:.1f}s")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(times_cpu, mag_cpu, 'o-', color='#1565C0', linewidth=2, markersize=4,
        label=f'MPS CPU (χ={CHI_CPU})')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.4, label='Thermal equilibrium')
ax.fill_between(times_cpu, 0, mag_cpu, alpha=0.08, color='#1565C0')
ax.set_xlabel('Time t (units of 1/Ω)', fontsize=13)
ax.set_ylabel('Staggered Magnetization M(t)', fontsize=13)
ax.set_title(f'Phase 1 (CPU): {N_ATOMS_CPU} atoms, χ={CHI_CPU}\n'
             f'Completed in {cpu_time:.1f}s', fontsize=14)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"💡 Notice the persistent oscillations — scarring prevents thermalization!")

---

## Phase 2 — GPU Mode (64 atoms, higher χ)

64 atoms, χ=64, 30 Trotter steps. Capture the full revival structure at higher accuracy.

👉 **[Start your free GPU trial →](https://maestro.qoroquantum.net)**

In [ ]:
N_ATOMS_GPU = 64
CHI_GPU = 64
N_STEPS_GPU = 30

print(f"Phase 2: {N_ATOMS_GPU} atoms, χ={CHI_GPU}, {N_STEPS_GPU} steps (GPU)")

observables_gpu = build_z_observables(N_ATOMS_GPU)
times_gpu = [0.0]
mag_gpu = [1.0]

t0 = time.time()
for step in range(1, N_STEPS_GPU + 1):
    qc = build_pxp_circuit(N_ATOMS_GPU, OMEGA, V, DT, step)
    res = qc.estimate(
        simulator_type=maestro.SimulatorType.CuQuantum,
        simulation_type=maestro.SimulationType.MatrixProductState,
        observables=observables_gpu,
        max_bond_dimension=CHI_GPU,
    )
    m = compute_staggered_magnetization(res['expectation_values'], N_ATOMS_GPU)
    times_gpu.append(step * DT)
    mag_gpu.append(m)
    if step % 10 == 0:
        print(f"  step {step:3d}  t={step*DT:.1f}  M(t)={m:.4f}")

gpu_time = time.time() - t0
print(f"\n✅ Phase 2 complete in {gpu_time:.1f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Phase 1
axes[0].plot(times_cpu, mag_cpu, 'o-', color='#1565C0', linewidth=2, markersize=3)
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.4)
axes[0].fill_between(times_cpu, 0, mag_cpu, alpha=0.08, color='#1565C0')
axes[0].set_title(f'Phase 1 — CPU: {N_ATOMS_CPU} atoms, χ={CHI_CPU}\n{cpu_time:.1f}s')
axes[0].set_xlabel('Time t (1/Ω)')
axes[0].set_ylabel('Staggered Magnetization M(t)')
axes[0].grid(alpha=0.3)

# Phase 2
axes[1].plot(times_gpu, mag_gpu, 'o-', color='#E91E63', linewidth=2, markersize=3)
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.4)
axes[1].fill_between(times_gpu, 0, mag_gpu, alpha=0.08, color='#E91E63')
axes[1].set_title(f'Phase 2 — GPU: {N_ATOMS_GPU} atoms, χ={CHI_GPU}\n{gpu_time:.1f}s')
axes[1].set_xlabel('Time t (1/Ω)')
axes[1].set_ylabel('Staggered Magnetization M(t)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('quantum_scarring.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n⚡ Phase 1 (CPU, {N_ATOMS_CPU} atoms): {cpu_time:.1f}s")
print(f"⚡ Phase 2 (GPU, {N_ATOMS_GPU} atoms): {gpu_time:.1f}s")
print(f"\nThe persistent oscillations are quantum scars — ETH violation!")

---

## Ready for Large-Scale Scarring?

GPU mode makes 64+ atom scarring simulations practical — full revival structure without overnight runs.

👉 **[Start your free GPU trial at maestro.qoroquantum.net](https://maestro.qoroquantum.net)**